# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sarahibdah/Flyrank-ML-Internship-Sarah/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Scoring, built on a classification core. At the base, my question is classification: for each page, yes or no — will its traffic fade in the next 30 days? Every historical page already carries this answer in its data (that's what makes it supervised). But a yes/no flag alone doesn't serve the editor: 54% of pages are declining, so a flag would fire everywhere. The real decision is which pages get attention first, and that needs an order. So I frame the task as scoring: the model outputs a probability of fading per page, and I sort by it into a refresh calendar. Not clustering — I already know the groups I care about (fading soon vs not); though I may use a small clustering step later to explore trajectory shapes, it isn't the core task.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

What I really want to predict is "this page needs a refresh now to avoid fading" - but that can't be measured directly, so I use a proxy I can measure: did the page's impressions in the next 30 days drop by 20% or more compared to the previous 30 days? Yes = 1, no = 0. I'm naming it out loud: this is a proxy, and the 20% threshold is a first guess I'll revisit with the daily warehouse data.**bold text**

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: precision@K, decided now, before any training. If my refresh calendar tells editors to work on K pages this month, precision@K = how many of those K truly went on to fade. I'll validate it the honest way for time data: train on earlier months, test on later months the model never saw (not a random split), and always compare against the base rate and against a simple rule baseline.

## 4. The unit of analysis, as a real dataframe





*Load your lane's slice and show it: one row = one what?*

One row = one page: 30,000 rows and 30,000 unique content_ids, so each row is one published article on a client's website, summarized over 90 days. There are 32 clients, so each website contributes many pages — the page is the unit, the client is context and a validation fence. The code also sketches my target: prev_30d as "the past", last_30d as "the future", label = impressions dropped 20% or more, only for pages with at least 100 prior impressions (a drop from 3 to 2 is noise, not fading). The label compares each page to its own past — change, not absolute traffic. Caveat: in this table both windows are already in the past, so this is only a shape-sketch; the real target needs the daily warehouse data with a true future window, where my grain becomes one page-window instead of one page.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/sarahibdah/Flyrank-ML-Internship-Sarah/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# The unit of analysis: one row = one page (a 90-day summary of that page)
print(f"{len(df):,} rows = {df['content_id'].nunique():,} pages")
display(df[["content_id", "client_id", "content_age_days",
            "impressions_prev_30d", "impressions_last_30d"]].head())

# Sketch of my target: did impressions drop >=20% month over month?
# (toy version - the real one uses daily warehouse data and a proper future window)
vol = df[df["impressions_prev_30d"] >= 100].copy()   # need volume to measure a drop
change = (vol["impressions_last_30d"] - vol["impressions_prev_30d"]) / vol["impressions_prev_30d"]
vol["target_sketch_faded"] = (change <= -0.20).astype(int)

print(f"\nPages with enough volume to measure: {len(vol):,}")
print(f"Share labeled 'faded' in this sketch: {vol['target_sketch_faded'].mean():.1%}")
display(vol[["content_id", "impressions_prev_30d", "impressions_last_30d", "target_sketch_faded"]].head())

30,000 rows = 30,000 pages


,content_id,client_id,content_age_days,impressions_prev_30d,impressions_last_30d
0,content_304f48230142,client_f369cb89fc,187,987,578
1,content_a1fb4e703a9e,client_4e07408562,445,5915,2501
2,content_9aa793d4d895,client_7f2253d7e2,141,6089,2382
3,content_331d6c4de07b,client_19581e27de,463,4206,3626
4,content_d99b7a2d90ca,client_3fdba35f04,263,6452,4211



Pages with enough volume to measure: 18,010
Share labeled 'faded' in this sketch: 61.6%


,content_id,impressions_prev_30d,impressions_last_30d,target_sketch_faded
0,content_304f48230142,987,578,1
1,content_a1fb4e703a9e,5915,2501,1
2,content_9aa793d4d895,6089,2382,1
3,content_331d6c4de07b,4206,3626,0
4,content_d99b7a2d90ca,6452,4211,1


## 5. Why ML beats a fixed rule here

> Add blockquote



*What makes the pattern too messy for an if-statement?*

A fixed rule holds one threshold, like "flag pages older than 180 days" or "flag pages with fewer than 500 impressions." But my week-1 numbers showed that intuition is backwards: pages 31-90 days old decline at 66.9% while pages over a year old decline at 42.6%. The pattern is real but it isn't a single threshold — the shape of decay depends on age, content type, client, and more, all tangled together. That's exactly the case where a model earns its place. Also, the classic hand rule fires on only 17 of 30,000 pages — precise but nearly useless for coverage. And my sketch shows 61.6% of measurable pages are fading, so the real job is ranking which pages first, not just flagging yes/no. A rule can't produce an order; a model outputs a probability per page that I can sort into an editor's queue.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.